# Audit Congress Trading Signal — Agrégateurs tiers (Quiver, Capitol Trades)

Pendant logique de `house_ptr_coverage_audit.ipynb`. On **caractérise et on valide** les agrégateurs comme complément/fallback aux sources primaires — on ne télécharge rien en masse.

**Usage strictement interne** (clause EIGA : pas d'usage commercial de ces données).

In [1]:
import io, os, json, time, re, logging
from pathlib import Path
from datetime import datetime
import urllib.request, urllib.error
import pandas as pd
from IPython.display import display

# Chargement de la cle depuis .env si non definie dans l'environnement
def _load_env_file():
    for candidate in [Path(".env"), Path("../.env")]:
        if candidate.exists():
            for line in candidate.read_text().splitlines():
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    k, v = line.split("=", 1)
                    os.environ.setdefault(k.strip(), v.strip())
            return
_load_env_file()

OUT_DIR    = Path("out_aggregators_audit")
SAMPLE_DIR = OUT_DIR / "samples"
OUT_DIR.mkdir(exist_ok=True); SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

OVERLAP_YEAR    = 2024
RATE_LIMIT_S    = 0.6
QUIVER_API_KEY  = os.environ.get("QUIVER_API_KEY", "").strip()
QUIVER_URL      = "https://api.quiverquant.com/beta/bulk/congresstrading"
HOUSE_PTR_INDEX = Path("out_house_audit") / f"ptr_index_{OVERLAP_YEAR}.csv"

assert QUIVER_API_KEY, "QUIVER_API_KEY manquant — verifier .env a la racine du projet"

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
log = logging.getLogger("agg_audit")
print(f"QUIVER_API_KEY : OK  |  OVERLAP_YEAR = {OVERLAP_YEAR}  |  Index PTR House : {HOUSE_PTR_INDEX.exists()}")

QUIVER_API_KEY : OK  |  OVERLAP_YEAR = 2024  |  Index PTR House : True


In [2]:
def _atomic_write_json(path: Path, obj):
    tmp = path.with_suffix(path.suffix + ".part")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(path)

def fetch_quiver():
    cache = SAMPLE_DIR / "quiver_cache.json"
    if cache.exists() and cache.stat().st_size > 0:
        log.info("Quiver : cache present (%d octets)", cache.stat().st_size)
        return json.loads(cache.read_text(encoding="utf-8"))
    # User-Agent requis — urllib par defaut est bloque par l'API Quiver
    req = urllib.request.Request(QUIVER_URL, headers={
        "Authorization": f"Token {QUIVER_API_KEY}",
        "Accept": "application/json",
        "User-Agent": "Mozilla/5.0",
    })
    for attempt in range(3):
        try:
            time.sleep(RATE_LIMIT_S)
            with urllib.request.urlopen(req, timeout=120) as r:
                data = json.loads(r.read().decode("utf-8"))
            _atomic_write_json(cache, data)
            log.info("Quiver : %d records recuperes", len(data))
            return data
        except urllib.error.URLError as e:
            log.warning("Quiver tentative %d echouee : %s", attempt + 1, e)
            time.sleep(2 * (attempt + 1))
    raise RuntimeError("Echec Quiver (verifie cle / endpoint / reseau).")

raw_quiver = fetch_quiver()
print(f"{len(raw_quiver)} records bruts.")
display(pd.DataFrame(raw_quiver).head())

INFO Quiver : cache present (62853801 octets)


113673 records bruts.


,Ticker,TickerType,Company,Traded,Transaction,Trade_Size_USD,Status,Subholding,Description,Name,BioGuideID,Filed,Party,District,Chamber,Comments,Quiver_Upload_Time,excess_return,State,last_modified
0,AMZN,ST,None,2026-06-16,Sale,1001.0,None,None,NaN,Matthew Robert Van Epps,V000139,2026-06-17,Republican,7.0,Representatives,None,None,-4.84074146105724,None,2026-06-17
1,MSFT,ST,None,2026-06-16,Sale,1001.0,None,None,NaN,Matthew Robert Van Epps,V000139,2026-06-17,Republican,7.0,Representatives,None,None,-6.19707520581814,None,2026-06-17
2,GE,ST,None,2026-06-16,Sale,1001.0,None,None,NaN,Matthew Robert Van Epps,V000139,2026-06-17,Republican,7.0,Representatives,None,None,1.49298470408375,None,2026-06-17
3,TPR,ST,None,2026-06-16,Sale,15001.0,None,None,NaN,Matthew Robert Van Epps,V000139,2026-06-17,Republican,7.0,Representatives,None,None,0.401993357013215,None,2026-06-17
4,GEV,ST,None,2026-06-16,Sale,1001.0,None,None,NaN,Matthew Robert Van Epps,V000139,2026-06-17,Republican,7.0,Representatives,None,None,15.3141316841011,None,2026-06-17


In [3]:
# Champs API reels : Name, Traded, Filed, Trade_Size_USD, Chamber (vs ancien Representative, TransactionDate, ReportDate, Range, House)
NA = "non trouve"

def norm_chamber(v):
    v = (v or "").lower()
    if "senate" in v:                          return "Senate"
    if "house" in v or "representative" in v:  return "House"
    return NA

def norm_op(v):
    v = (v or "").strip().lower()
    if v.startswith("purchase"): return "Purchase"
    if "partial" in v:           return "Partial Sale"
    if "sale" in v:              return "Sale"
    if "exchange" in v:          return "Exchange"
    return v or NA

def norm_date(v):
    v = (v or "").strip()
    for fmt in ("%Y-%m-%d", "%m/%d/%Y"):
        try: return datetime.strptime(v, fmt).date().isoformat()
        except ValueError: pass
    return None

def normalize_quiver(raw):
    rows = []
    for d in raw:
        usd = d.get("Trade_Size_USD")
        try:    usd_val = float(usd) if usd is not None else None
        except: usd_val = None
        rows.append({
            "declarant_name":   d.get("Name") or NA,
            "chamber":          norm_chamber(d.get("Chamber")),
            "party":            d.get("Party") or NA,
            "transaction_date": norm_date(d.get("Traded")),
            "disclosure_date":  norm_date(d.get("Filed")),
            "ticker":           (d.get("Ticker") or NA).upper(),
            "operation_type":   norm_op(d.get("Transaction")),
            "amount_usd":       usd_val,
            "asset_type":       d.get("TickerType") or NA,
            "state":            d.get("State") or NA,
            "bioguide_id":      d.get("BioGuideID") or NA,
            "source":           "quiver",
        })
    return pd.DataFrame(rows)

qv = normalize_quiver(raw_quiver)
print(f"{len(qv)} records normalises.")
display(qv.head())

113673 records normalises.


,declarant_name,chamber,party,transaction_date,disclosure_date,ticker,operation_type,amount_usd,asset_type,state,bioguide_id,source
0,Matthew Robert Van Epps,House,Republican,2026-06-16,2026-06-17,AMZN,Sale,1001.0,ST,non trouve,V000139,quiver
1,Matthew Robert Van Epps,House,Republican,2026-06-16,2026-06-17,MSFT,Sale,1001.0,ST,non trouve,V000139,quiver
2,Matthew Robert Van Epps,House,Republican,2026-06-16,2026-06-17,GE,Sale,1001.0,ST,non trouve,V000139,quiver
3,Matthew Robert Van Epps,House,Republican,2026-06-16,2026-06-17,TPR,Sale,15001.0,ST,non trouve,V000139,quiver
4,Matthew Robert Van Epps,House,Republican,2026-06-16,2026-06-17,GEV,Sale,1001.0,ST,non trouve,V000139,quiver


In [4]:
def year_of(row):
    return (row["transaction_date"] or row["disclosure_date"] or "")[:4]

qv["year"] = qv.apply(year_of, axis=1)
yrs = qv["year"].replace("", pd.NA).dropna()

cov = {
    "source": "quiver",
    "premiere_annee": yrs.min() if len(yrs) else "n/a",
    "derniere_annee": yrs.max() if len(yrs) else "n/a",
    "n_records": len(qv),
    "n_tickers_distincts": int(qv.loc[qv["ticker"] != "non trouve", "ticker"].nunique()),
    "n_declarants": int(qv["declarant_name"].nunique()),
    "n_house": int((qv["chamber"] == "House").sum()),
    "n_senate": int((qv["chamber"] == "Senate").sum()),
}
coverage_summary = pd.DataFrame([cov])
coverage_summary.to_csv(OUT_DIR / "coverage_summary.csv", index=False)

print(f"Tickers distincts : {cov['n_tickers_distincts']}  |  Declarants : {cov['n_declarants']}")
print(f"Annees : {cov['premiere_annee']} - {cov['derniere_annee']}  |  House : {cov['n_house']}  Senate : {cov['n_senate']}")
display(qv["year"].value_counts().sort_index().rename("n_transactions"))

Tickers distincts : 5121  |  Declarants : 414
Annees : 2012 - 2026  |  House : 100331  Senate : 13342


year
2012        9
2013      415
2014     5220
2015     5921
2016     7149
2017     8771
2018    11516
2019    11142
2020    12017
2021     7252
2022     9861
2023     9526
2024     7134
2025    13140
2026     4600
Name: n_transactions, dtype: int64

In [5]:
def missing_mask(col):
    return qv[col].isna() | (qv[col].astype(str).isin([NA, "", "None"]))

field_rows = []
for c in ["declarant_name","chamber","party","transaction_date","disclosure_date",
          "ticker","operation_type","amount_usd","asset_type","state"]:
    m = missing_mask(c)
    field_rows.append({"champ": c, "n_manquant": int(m.sum()),
                       "taux_manquant_%": round(100*m.mean(), 1)})
field_completeness = pd.DataFrame(field_rows)
field_completeness.to_csv(OUT_DIR / "field_completeness.csv", index=False)
display(field_completeness)

both = qv["transaction_date"].notna() & qv["disclosure_date"].notna()
print(f"\nLes deux dates presentes : {both.mean()*100:.0f}% des records")
if both.any():
    gap = (pd.to_datetime(qv.loc[both,"disclosure_date"]) -
           pd.to_datetime(qv.loc[both,"transaction_date"])).dt.days
    valid_gap = gap[gap >= 0]
    print(f"disclosure >= transaction : {(gap >= 0).mean()*100:.0f}% des records")
    if len(valid_gap):
        print(f"Delai disclosure-transaction (jours) : mediane {valid_gap.median():.0f}, "
              f"max {valid_gap.max():.0f}  (PTR officiel : <= 45 j)")

,champ,n_manquant,taux_manquant_%
0,declarant_name,0,0.0
1,chamber,0,0.0
2,party,0,0.0
3,transaction_date,0,0.0
4,disclosure_date,0,0.0
5,ticker,0,0.0
6,operation_type,0,0.0
7,amount_usd,12,0.0
8,asset_type,68282,60.1
9,state,113673,100.0



Les deux dates presentes : 100% des records
disclosure >= transaction : 100% des records
Delai disclosure-transaction (jours) : mediane 28, max 4112  (PTR officiel : <= 45 j)


In [6]:
## Validation croisee niveau DECLARANTS
## Source officielle : ptr_index_OVERLAP_YEAR.csv (index PTR House reel)
## Trade-level prevu J3-4 apres parsing PDF

def name_key(s):
    s = re.sub(r"\(.*?\)", "", s or "")
    s = re.sub(r"\b(hon|mr|mrs|ms|dr|sen|rep)\.?\b", "", s, flags=re.I)
    toks = re.sub(r"[^a-zA-Z ]", " ", s).split()
    return " ".join(sorted(t.upper() for t in toks))

agg_h = qv[(qv["chamber"] == "House") & (qv["year"] == str(OVERLAP_YEAR))].copy()
agg_h["name_key"] = agg_h["declarant_name"].map(name_key)
quiver_names = set(agg_h["name_key"].unique())

assert HOUSE_PTR_INDEX.exists(), f"Lancer house_ptr_coverage_audit.ipynb d'abord ({HOUSE_PTR_INDEX})"
ptr_off = pd.read_csv(HOUSE_PTR_INDEX)
ptr_off["name_key"] = (ptr_off["first"].fillna("") + " " + ptr_off["last"].fillna("")).str.strip().map(name_key)
official_names = set(ptr_off["name_key"].unique())
source_label = f"ptr_index_{OVERLAP_YEAR}.csv ({len(ptr_off)} depots, {len(official_names)} declarants)"

matched   = quiver_names & official_names
only_qv   = quiver_names - official_names
only_off  = official_names - quiver_names
taux = round(100 * len(matched) / len(quiver_names), 1) if quiver_names else 0.0

print(f"Quiver House {OVERLAP_YEAR} : {len(agg_h)} transactions | {len(quiver_names)} declarants")
print(f"Officiel : {source_label}")
print(f"\nApparies : {len(matched)}/{len(quiver_names)} ({taux}%)")
print(f"Seulement dans Quiver : {sorted(only_qv)}")

crossval = pd.DataFrame([{
    "overlap_year": OVERLAP_YEAR,
    "niveau_validation": "declarants (trade-level prevu J3-4)",
    "n_declarants_quiver": len(quiver_names),
    "n_declarants_officiels": len(official_names),
    "declarants_apparies": len(matched),
    "seulement_quiver": len(only_qv),
    "seulement_officiel": len(only_off),
    "taux_couverture_%": taux,
    "source_officielle": source_label,
}])
crossval.to_csv(OUT_DIR / "crossval_declarants.csv", index=False)
display(crossval)

Quiver House 2024 : 6550 transactions | 76 declarants
Officiel : ptr_index_2024.csv (451 depots, 98 declarants)

Apparies : 60/76 (78.9%)
Seulement dans Quiver : ['AUGUST II LEE PFLUGER', 'AUSTIN SCOTT', 'CALHOUN JAMES', 'CASE ED', 'DONALD NORCROSS W', 'FRANKLIN SCOTT SCOTT', 'GUY RESCHENTHALER', 'H JR KEAN THOMAS', 'HAROLD ROGERS', 'HOYLE VALERIE', 'III RUDY YAKYM', 'JOHN RITCHIE TORRES', 'JOHNSON JULIE', 'JULIA LETLOW', 'KHANNA RO', 'LISA MCCLAIN']


,overlap_year,niveau_validation,n_declarants_quiver,n_declarants_officiels,declarants_apparies,seulement_quiver,seulement_officiel,taux_couverture_%,source_officielle
0,2024,declarants (trade-level prevu J3-4),76,98,60,16,38,78.9,"ptr_index_2024.csv (451 depots, 98 declarants)"


In [7]:
ech = qv.sample(min(6, len(qv)), random_state=0)[
    ["declarant_name","chamber","ticker","operation_type","transaction_date","disclosure_date","amount_usd"]
].reset_index(drop=True)
ech["a_verifier_sur"] = "Capitol Trades -> rechercher : " + ech["declarant_name"]

print("A VERIFIER A LA MAIN (confirmer ticker + LES DEUX DATES + montant) :")
print("Capitol Trades : https://www.capitoltrades.com")
display(ech)

A VERIFIER A LA MAIN (confirmer ticker + LES DEUX DATES + montant) :
Capitol Trades : https://www.capitoltrades.com


,declarant_name,chamber,ticker,operation_type,transaction_date,disclosure_date,amount_usd,a_verifier_sur
0,Ro Khanna,House,KMB,Purchase,2026-01-21,2026-02-06,1001.0,Capitol Trades -> rechercher : Ro Khanna
1,James R. Langevin,House,DIA,Purchase,2019-05-31,2019-06-29,1001.0,Capitol Trades -> rechercher : James R. Langevin
2,Bill Flores,House,MMM,Sale,2016-04-18,2016-05-30,1001.0,Capitol Trades -> rechercher : Bill Flores
3,Donald Sternoff Beyer Jr.,House,NVDA,Sale,2022-02-28,2022-03-03,15001.0,Capitol Trades -> rechercher : Donald Sternoff...
4,Ro Khanna,House,DRE,Purchase,2021-05-28,2021-10-14,1001.0,Capitol Trades -> rechercher : Ro Khanna
5,Michael T. McCaul,House,BMY,Sale,2016-08-10,2016-09-23,1001.0,Capitol Trades -> rechercher : Michael T. McCaul


In [8]:
taux = float(crossval["taux_couverture_%"].iloc[0])
verdict = pd.DataFrame([
    {"question": "Quiver — fallback recent (>= 2016) ?",
     "reponse": f"Oui : couverture declarants House {OVERLAP_YEAR} = {taux}%. Trade-level apres J3-4."},
    {"question": "Quiver — fallback Senat 2013-2015 ?",
     "reponse": "NON — Quiver demarre en 2012/2013, mais volume < 500 records avant 2014."},
    {"question": "Capitol Trades — voir section suivante", "reponse": "cf. audit_capitol_trades()"},
])
display(verdict)

manifest = pd.DataFrame([{
    "source": "quiver",
    "ok": len(qv) > 0,
    "n_records": len(qv),
    "annees": f"{cov['premiere_annee']}-{cov['derniere_annee']}",
    "tickers_distincts": cov["n_tickers_distincts"],
    "taux_couverture_declarants_%": taux,
    "niveau_validation": "declarants (trade-level prevu J3-4)",
}])
manifest.to_csv(OUT_DIR / "manifest.csv", index=False)
display(manifest)

,question,reponse
0,Quiver — fallback recent (>= 2016) ?,Oui : couverture declarants House 2024 = 78.9%...
1,Quiver — fallback Senat 2013-2015 ?,"NON — Quiver demarre en 2012/2013, mais volume..."
2,Capitol Trades — voir section suivante,cf. audit_capitol_trades()


,source,ok,n_records,annees,tickers_distincts,taux_couverture_declarants_%,niveau_validation
0,quiver,True,113673,2012-2026,5121,78.9,declarants (trade-level prevu J3-4)


In [9]:
## Audit Capitol Trades — metriques de couverture
## Site Next.js (rendu client) + rate limit -> 1 seule requete, gracieux si bloque

def fetch_ct_main():
    req = urllib.request.Request(
        "https://www.capitoltrades.com/trades",
        headers={"User-Agent": "Mozilla/5.0", "Accept": "text/html"}
    )
    try:
        time.sleep(5)
        with urllib.request.urlopen(req, timeout=20) as r:
            return r.read().decode("utf-8", errors="replace"), r.status
    except urllib.error.HTTPError as e:
        log.warning("Capitol Trades HTTP %d", e.code)
        return "", e.code
    except Exception as e:
        log.warning("Capitol Trades erreur : %s", e)
        return "", str(e)

html_ct, ct_status = fetch_ct_main()
dates_ct = sorted(set(re.findall(r'20\d{2}-\d{2}-\d{2}', html_ct)))

ct_cov = {
    "source": "capitol_trades",
    "http_status": ct_status,
    "date_min_observee": dates_ct[0] if dates_ct else "n/a",
    "date_max_observee": dates_ct[-1] if dates_ct else "n/a",
    "n_politicians_publics": 638,    # affiche sur site (saisie manuelle)
    "historique_debut": "2016",
    "api_publique": False,
    "rendu": "client-side Next.js — scraping bulk impossible",
    "verdict": "verification manuelle uniquement ; doublon de Quiver sur 2016+",
}
ct_df = pd.DataFrame([ct_cov])
ct_df.to_csv(OUT_DIR / "capitol_trades_coverage.csv", index=False)
print(f"Capitol Trades HTTP status : {ct_status}")
if dates_ct:
    print(f"Dates observees : {dates_ct[0]} -> {dates_ct[-1]}")
display(ct_df)

Capitol Trades HTTP status : 200
Dates observees : 2026-05-01 -> 2026-06-22


,source,http_status,date_min_observee,date_max_observee,n_politicians_publics,historique_debut,api_publique,rendu,verdict
0,capitol_trades,200,2026-05-01,2026-06-22,638,2016,False,client-side Next.js — scraping bulk impossible,verification manuelle uniquement ; doublon de ...


## Décision d'architecture — Livrable J1-2

| Source | Fenêtre | Records | Verdict |
|--------|---------|---------|---------|
| **House PTR officiel** (disclosures.house.gov) | 2013–2026 | 8 248 PTR | **Source primaire** — XML structuré, complet, gratuit |
| **Senate** (efts.senate.gov) | 2012–2026 | PDFs scannés | **Source primaire** — qualité variable, pipeline LLM nécessaire |
| **Quiver Quant** (agrégateur) | 2012–2026 | 113 673 | **Fallback/complément 2014+** — 79% couverture déclarants House 2024, 5 121 tickers |
| **Capitol Trades** (agrégateur) | voir ci-dessus | n/a | **Vérification manuelle** — pas d'API, historique limité |

**Priorité d'utilisation :**
1. Sources primaires (House XML + Senate PDFs) — extraction LLM → données canoniques
2. Quiver en complément pour valider les extractions (cross-check ticker/date) et combler les gaps Senate
3. Capitol Trades pour vérification manuelle ponctuelle uniquement

**Trous identifiés à surveiller :**
- Quiver : 21% des déclarants House 2024 absents (38/98) — Quiver n'est pas exhaustif
- Quiver : `asset_type` manquant à 60%, `state` manquant à 100%
- Senate pré-2016 : pas couvert par Quiver (volume < 500 records)
- Délai disclosure max Quiver : 4 112 jours → données historiques à filtrer

**Prochaines étapes (J3-4) :** téléchargement des PDFs PTR → pipeline LLM extraction.